# 周波数依存性 (Lakeshore155) LI5650 1台

In [ ]:
import logging
from W155_Socket import LakeShore155, generate_current, generate_Prime_num
from LI5660_Socket import NF_LI5600
from FileOperation import FileOperation, setup_logger
FO = FileOperation()
import time
import pandas as pd
import sys
import matplotlib.pyplot as plt
import gc

W155_VISA   = "TCPIP0::100.100.1.155::7777::SOCKET"
LI5660_VISA = "TCPIP0::100.100.1.55::5025::SOCKET"
# LI5660_VISA = "TCPIP0::192.168.1.56::5025::SOCKET"

Filename, Infoname, Figname, Logname = FO.Make_file()
#*************************************************************************************************
level = 0.1    # mA
start_f   = 71  # Hz
Harmonics = [1, 2, 3]
Averaging = [50, 100, 100]
frequency_list = generate_Prime_num(start_f, 100)
Waittime     = [10 for _ in range(len(frequency_list))]  # sec
terminal = "REAR"  # "REAR" or "FRONT"
parameters_mode = "AUTO"  # "AUTO", "MANUAL", "USER", "INFO"
parameters_table = None
Log_level = logging.INFO    # INFO, DEBUG
#*************************************************************************************************

logger    = setup_logger(log_file=Logname, level=Log_level)
LI_logger = setup_logger(log_file=Logname, logger_name="LI5600_1", level=Log_level)
# LI_logger = setup_logger(log_file=Logname, logger_name="LI5600_2", level=Log_level)
LS_logger  = setup_logger(log_file=Logname, logger_name="LakeShore155", level=Log_level)
LIQ_logger = setup_logger(log_file=Logname, logger_name="LI5600_Q", level=Log_level)

if parameters_mode in ("USER", "INFO"):
     parameters_table = FO.Read_parameter()
start_time = time.time()
Data = pd.DataFrame([]); Info = pd.DataFrame([])


with LakeShore155(W155_VISA, logger=LS_logger) as W155:
    W155.AC_initialize(level, start_f,terminal)
    W155.output_ON()
    with NF_LI5600(LI5660_VISA, logger=LI_logger) as LI5600:
        LI5600.initialize_instrument(terminal= "A", Reference= "RINP", RefType = "TPOS", Notch ="ON", Filter ="MOV", Ground = "FLO")
    W155.Long_Wait(30)
    for i, frequency in enumerate(frequency_list):
        W155.clear_log()
        W155.set_frequency(frequency)
        Freq = W155.get_frequency()
        Freq_pd = pd.DataFrame([Freq], columns=["Frequency(Hz)"])
        W155.Long_Wait(Waittime[i])

        while True:
            try:
                with NF_LI5600(LI5660_VISA, logger=LI_logger) as LI5600:
                    Result, parameters = LI5600.harmonic_measurement_loop(Harmonics=Harmonics, Averaging=Averaging, start_time=start_time, mode=parameters_mode, Manual_parameter=parameters_table, frequency=frequency)
                    parameters_freq = pd.concat([Freq_pd, parameters], axis=1)
                    Results = pd.concat([Freq_pd, Result], axis=1)
                    Data    = pd.concat([Data, Results], ignore_index=True)
                    Info = pd.concat([Info, parameters_freq], ignore_index=True)

                # fig = FO.plot_figure(Data, Harmonics)
                fig, axs = FO.plot_figure(Data, Harmonics)
                fig.savefig(Figname)
                plt.close(fig)
                FileOperation.SaveFile(Data, Filename)
                FileOperation.SaveFile(Info, Infoname)

                if (i + 1) % 50 == 0:
                    gc.collect()
                break  # 成功したのでループを抜ける

            except IOError as e:
                        LI_logger.critical(f"IOError at index {i}, freq={frequency}: {e}")
                        W155.output_OFF()
                        sys.exit(1)

            except Exception as e:
                LI_logger.error(f"LI5600 measurement failed at index {i} (Frequency = {frequency}): {e}")
                time.sleep(10)  # 少し待ってから再試行

    W155.output_OFF()

display(Data)

# 周波数依存性 (Lakeshore155) LI5650 2台

In [ ]:
import logging
import time
import sys
import gc
import pandas as pd
import matplotlib.pyplot as plt

from W155_Socket import LakeShore155
from LI5660_Socket import NF_LI5600
from LI5660_Socket import NF_LI5660_Q
from W155_Socket import generate_current, generate_Prime_num
from FileOperation import FileOperation, setup_logger
FO = FileOperation()

W155_VISA   = "TCPIP0::100.100.1.155::7777::SOCKET"
LI5660_VISA1 = "TCPIP0::100.100.1.55::5025::SOCKET"
LI5660_VISA2 = "TCPIP0::100.100.1.56::5025::SOCKET"

Filename, Infoname, Figname, Logname = FO.Make_file()
#*************************************************************************************************
level = 0.1    # mA
start_f   = 71  # Hz
Harmonics = [1, 2, 3]
Averaging = [50, 100, 100]
frequency_list = generate_Prime_num(start_f, 100)
Waittime     = [10 for _ in range(len(frequency_list))]  # sec
terminal = "REAR"  # "REAR" or "FRONT"
parameters_mode = "AUTO"  # "AUTO", "MANUAL", "USER", "INFO"
parameters_table = None
Log_level = logging.INFO    # INFO, DEBUG
#*************************************************************************************************

logger    = setup_logger(log_file=Logname)
# LI1_logger = setup_logger(log_file=Logname, logger_name="LI5600_1")
# LI2_logger = setup_logger(log_file=Logname, logger_name="LI5600_2")
LS_logger  = setup_logger(log_file=Logname, logger_name="LakeShore155", level=Log_level)
LIQ_logger = setup_logger(log_file=Logname, logger_name="LI5600_Q", level=Log_level)

if parameters_mode in ("USER", "INFO"):
    parameters_table = FO.Read_parameter()

# 初期設定
li_init_kwargs = dict(terminal="A",  Reference="RINP", RefType="TPOS", Notch="ON", Filter="MOV", Ground="FLO")
LIQ = NF_LI5660_Q(visas=[LI5660_VISA1, LI5660_VISA2], logger=LIQ_logger, init_kwargs=li_init_kwargs, retry_wait=10.0)

Data = pd.DataFrame([])
Info = pd.DataFrame([])
start_time = time.time()

with LakeShore155(W155_VISA, logger=LS_logger) as W155:
    W155.AC_initialize(level, start_f, terminal)
    W155.output_ON()
    W155.Long_Wait(10)

    for i, frequency in enumerate(frequency_list):
        W155.clear_log()
        W155.set_frequency(frequency)
        Freq = W155.get_frequency()
        Freq_pd = pd.DataFrame([Freq], columns=["Frequency(Hz)"])
        W155.Long_Wait(Waittime[i])

        # 並列測定呼び出し
        Result, parameters = LIQ.measure(Harmonics=Harmonics, Averaging=Averaging, start_time=start_time, mode=parameters_mode, Manual_parameter=parameters_table, frequency=frequency )
        Results = pd.concat([Freq_pd, Result], axis=1)
        parameters_freq = pd.concat([Freq_pd, parameters], axis=1)
        Data = pd.concat([Data, Results], ignore_index=True)
        Info = pd.concat([Info, parameters_freq], ignore_index=True)

        fig, axs = FO.plot_figure(Data, Harmonics)
        fig.savefig(Figname)
        plt.close(fig)
        FO.SaveFile(Data, Filename)
        FO.SaveFile(Info, Infoname)

        if (i + 1) % 50 == 0:
            gc.collect()

    W155.output_OFF()

# PPMS

In [ ]:
import logging
import MultiPyVu as mpv
import time
import sys
import gc
import pandas as pd
import matplotlib.pyplot as plt

from W155_Socket import LakeShore155
from LI5660_Socket import NF_LI5600
from LI5660_Socket import NF_LI5660_Q
from W155_Socket import generate_current, generate_Prime_num
from FileOperation import FileOperation, setup_logger
from PPMS import PPMS, Make_Sequence

FO = FileOperation()

W155_VISA    = "TCPIP0::100.100.1.155::7777::SOCKET"
LI5660_VISA1 = "TCPIP0::100.100.1.55::5025::SOCKET"
LI5660_VISA2 = "TCPIP0::100.100.1.56::5025::SOCKET"

Filename, Infoname, Figname, Logname = FO.Make_file(PPMS_dat=True)

# *************************************************************************************************
level = 0.1    # mA
frequency = 71  # Hz
Harmonics = [1, 2, 3]
Averaging = [50, 100, 100]
Hi = [1000, -1000]
H_step = [1000]
Temp = [300]
T_step = []
wait_T = [10]

Field_Sequence = Make_Sequence(Hi=Hi, H_step=H_step, kind="Field", loop=True)
Temp_Sequence = Make_Sequence(Hi=Temp, H_step=T_step, kind="Temperature", loop=False)

terminal = "REAR"  # "REAR" or "FRONT"
parameters_mode = "AUTO"  # "AUTO", "MANUAL", "USER", "INFO"
parameters_table = None
Log_level = logging.DEBUG
# *************************************************************************************************

logger      = setup_logger(log_file=Logname)
LS_logger   = setup_logger(log_file=Logname, logger_name="LakeShore155", level=Log_level)
LIQ_logger  = setup_logger(log_file=Logname, logger_name="LI5600_Q", level=Log_level)
PPMS_logger = setup_logger(log_file=Logname, logger_name="PPMS", level=Log_level)

if parameters_mode in ("USER", "INFO"):
    parameters_table = FO.Read_parameter()

if len(wait_T) != len(Temp_Sequence):
    raise ValueError("len(wait_T) must match len(Temp_Sequence)")

li_init_kwargs = dict(terminal="A", Reference="RINP", RefType="TPOS", Notch="ON", Filter="MOV", Ground="FLO")
LIQ = NF_LI5660_Q( visas=[LI5660_VISA1, LI5660_VISA2], logger=LIQ_logger, init_kwargs=li_init_kwargs, retry_wait=10.0)

start_time = time.time()

with PPMS(logger=logger) as ppms:
    for temp, wait in zip(Temp_Sequence, wait_T):
        Data = pd.DataFrame()
        Info = pd.DataFrame()
        current_temp = ppms.set_temp(target_k=temp, rate_k_per_min=10.0, wait=True, delay_sec=wait)
        Filename_T, Infoname_T, Figname_T = FO.add_temp_to_filenames(Filename, Infoname, Figname, temp)

        with LakeShore155(W155_VISA, logger=LS_logger) as W155:
            W155.AC_initialize(level, frequency, terminal)
            W155.output_ON()
            W155.Long_Wait(10)

            for Field in Field_Sequence:
                current_field = ppms.set_field(Field, field_mode="persistent", wait=True)
                Field_pd = pd.DataFrame([[current_field, current_temp]], columns=["Field (Oe)", "Temp (K)"])
                Result, parameters = LIQ.measure(Harmonics=Harmonics, Averaging=Averaging, start_time=start_time, mode=parameters_mode, Manual_parameter=parameters_table, frequency=frequency)
                Results = pd.concat([Field_pd, Result], axis=1)
                parameters_freq = pd.concat([Field_pd, parameters], axis=1)
                Data = pd.concat([Data, Results], ignore_index=True)
                Info = pd.concat([Info, parameters_freq], ignore_index=True)

                fig, axs = FO.plot_figure(Data, Harmonics)
                fig.savefig(Figname_T)
                plt.close(fig)

                FO.SaveFile(Data, Filename_T, PPMS_dat=True)
                FO.SaveFile(Info, Infoname_T)

            W155.output_OFF()

In [ ]:
import logging
import MultiPyVu as mpv
import time
import sys
import gc
import pandas as pd
import matplotlib.pyplot as plt

from W155_Socket import LakeShore155
from LI5660_Socket import NF_LI5600
from LI5660_Socket import NF_LI5660_Q
from W155_Socket import generate_current, generate_Prime_num
from FileOperation import FileOperation, setup_logger
from PPMS import PPMS
FO = FileOperation()

W155_VISA   = "TCPIP0::100.100.1.155::7777::SOCKET"
LI5660_VISA1 = "TCPIP0::100.100.1.55::5025::SOCKET"
LI5660_VISA2 = "TCPIP0::100.100.1.56::5025::SOCKET"

Filename, Infoname, Figname, Logname = FO.Make_file()
#*************************************************************************************************
level = 0.1    # mA
frequency   = 71  # Hz
Harmonics = [1, 2, 3]
Averaging = [50, 100, 100]
terminal = "REAR"  # "REAR" or "FRONT"
parameters_mode = "AUTO"  # "AUTO", "MANUAL", "USER", "INFO"
parameters_table = None
Log_level = logging.DEBUG    # INFO, DEBUG
#*************************************************************************************************

logger    = setup_logger(log_file=Logname)
# LI1_logger = setup_logger(log_file=Logname, logger_name="LI5600_1")
# LI2_logger = setup_logger(log_file=Logname, logger_name="LI5600_2")
LS_logger  = setup_logger(log_file=Logname, logger_name="LakeShore155", level=Log_level)
LIQ_logger = setup_logger(log_file=Logname, logger_name="LI5600_Q", level=Log_level)
PPMS_logger = setup_logger(log_file=Logname, logger_name="PPMS", level=Log_level)


if parameters_mode in ("USER", "INFO"):
    parameters_table = FO.Read_parameter()

# 初期設定
li_init_kwargs = dict(terminal="A",  Reference="RINP", RefType="TPOS", Notch="ON", Filter="MOV", Ground="FLO")
LIQ = NF_LI5660_Q(visas=[LI5660_VISA1, LI5660_VISA2], logger=LIQ_logger, init_kwargs=li_init_kwargs, retry_wait=10.0)

Data = pd.DataFrame([])
Info = pd.DataFrame([])
start_time = time.time()


with LakeShore155(W155_VISA, logger=LS_logger) as W155:
    W155.AC_initialize(level, frequency, terminal)
    W155.output_ON()
    W155.Long_Wait(10)

    Field = 100
    current_field = ppms.set_field(Field, field_mode="persistent", wait=True)
    Field_pd = pd.DataFrame([current_field], columns=["Field (Oe)"])

    # 並列測定呼び出し
    Result, parameters = LIQ.measure(Harmonics=Harmonics, Averaging=Averaging, start_time=start_time, mode=parameters_mode, Manual_parameter=parameters_table, frequency=frequency )
    Results = pd.concat([Field_pd, Result], axis=1)
    parameters_freq = pd.concat([Field_pd, parameters], axis=1)
    Data = pd.concat([Data, Results], ignore_index=True)
    Info = pd.concat([Info, parameters_freq], ignore_index=True)

    fig, axs = FO.plot_figure(Data, Harmonics)
    fig.savefig(Figname)
    plt.close(fig)
    FO.SaveFile(Data, Filename)
    FO.SaveFile(Info, Infoname)

    # if (i + 1) % 50 == 0:
    #     gc.collect()

    W155.output_OFF()

2026-04-08 17:28:02 [DEBUG] LakeShore155: Connected to TCPIP0::100.100.1.155::7777::SOCKET
2026-04-08 17:28:02 [INFO] LakeShore155: Starting AC initialization...
2026-04-08 17:28:02 [INFO] LakeShore155: >> *RST
2026-04-08 17:28:02 [DEBUG] LakeShore155: >> SOUR:FUNC:MODE CURRENT
2026-04-08 17:28:02 [DEBUG] LakeShore155: >> ROUT:TERM REAR
2026-04-08 17:28:02 [DEBUG] LakeShore155: >> SOUR:CURR:PROT 50.000000
2026-04-08 17:28:02 [DEBUG] LakeShore155: >> SOUR:FUNC SIN
2026-04-08 17:28:02 [DEBUG] LakeShore155: >> SOUR:FREQ 71.000000
2026-04-08 17:28:02 [DEBUG] LakeShore155: >> SOUR:CURR 1.000000e-04
2026-04-08 17:28:02 [INFO] LakeShore155: AC initialization complete (level = 0.1 mA, freq = 71 Hz)
2026-04-08 17:28:02 [INFO] LakeShore155: >> OUTP:STAT 1
2026-04-08 17:28:02 [INFO] LakeShore155: 待機時間: 10秒、開始時刻: 17時28分02秒
2026-04-08 17:28:12 [INFO] LakeShore155: >> OUTP:STAT 0
2026-04-08 17:28:12 [ERROR] LakeShore155: Failed to turn off output on exit: 'LakeShore155' object has no attribute 'outp